In [ ]:
%load_ext autoreload
%autoreload 2
import sys

sys.path.insert(0, "../")

In [ ]:
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError, conint

In [ ]:
load_dotenv("../.env")

## Load Data

In [ ]:
# Hand-labeled workbook incl. eHRAF text (not distributed; see README)
deities_df = pd.read_excel("../data/intermediate/TheTruthV7.xlsx", sheet_name="250_sampled_rows")
deities_df["Text"].head()

## Data Models

In [ ]:
Cert = conint(ge=0, le=100)

In [ ]:
Kind = Literal[
    "deity",
    "spirit",
    "ghost",
    "soul",
    "ancestor",
    "medium",
    "monster",
    "object",
    "place",
    "force",
    "other",
    "missing",
]
CatType = Literal["individual", "multiple", "missing"]
Gender = Literal["male", "female", "androgynous", "genderless", "missing"]

In [ ]:
class ExtractedElement(BaseModel):
    name: str
    kind: Kind = "missing"
    aliases: list[str] = Field(default_factory=list)
    cat_type: CatType = "missing"
    evidence_deity: list[str] = Field(default_factory=list)
    certainty_deity: Cert = 0


class LabeledElement(BaseModel):
    name: str
    gender: Gender = "missing"
    evidence_gender: list[str] = Field(default_factory=list)
    certainty_gender: Cert = 0

    # 20 binary cats
    cat_creator_universe: int = 0
    cat_creator_human: int = 0
    cat_mother: int = 0
    cat_wife: int = 0
    cat_primal: int = 0
    cat_omni: int = 0
    cat_present: int = 0
    cat_absent: int = 0
    cat_warrior: int = 0
    cat_nature: int = 0
    cat_cosmos: int = 0
    cat_death: int = 0
    cat_ruler: int = 0
    cat_dual: int = 0
    cat_trick: int = 0
    cat_evil: int = 0
    cat_good: int = 0
    cat_demigod: int = 0
    cat_inter: int = 0
    cat_object_force: int = 0


class ParagraphExtraction(BaseModel):
    non_english: int = 0
    translation_en: str
    inception_myth: int = 0
    elements: list[ExtractedElement]


class ParagraphLabels(BaseModel):
    inception_myth: int = 0
    labels: list[LabeledElement]

In [ ]:
class EnglishCheckResult(BaseModel):
    non_english: bool
    translation_en: str | None = None

In [ ]:
class ExtractedElement(BaseModel):
    name: str
    kind: Kind = "missing"
    aliases: list[str] = Field(default_factory=list)
    cat_type: CatType = "missing"
    evidence_deity: list[str] = Field(default_factory=list)
    certainty_deity: Cert = 0


class DeitiesExtraction(BaseModel):
    elements: list[ExtractedElement]

In [ ]:
class LabeledElement(BaseModel):
    name: str
    gender: Gender = "missing"
    evidence_gender: list[str] = Field(default_factory=list)
    certainty_gender: Cert = 0

    # 20 binary cats
    cat_creator_universe: int = 0
    cat_creator_human: int = 0
    cat_mother: int = 0
    cat_wife: int = 0
    cat_primal: int = 0
    cat_omni: int = 0
    cat_present: int = 0
    cat_absent: int = 0
    cat_warrior: int = 0
    cat_nature: int = 0
    cat_cosmos: int = 0
    cat_death: int = 0
    cat_ruler: int = 0
    cat_dual: int = 0
    cat_trick: int = 0
    cat_evil: int = 0
    cat_good: int = 0
    cat_demigod: int = 0
    cat_inter: int = 0
    cat_object_force: int = 0


class LabellingExtraction(BaseModel):
    inception_myth: int = Field(ge=0, le=1)
    labels: list[LabeledElement]

## OpenAI Call

In [ ]:
import json

In [ ]:
deity_text_sample = deities_df["Text"][1]
client = OpenAI()

In [ ]:
def call_structured(
    client: OpenAI,
    *,
    model: str = "gpt-4o-2024-08-06",
    system_prompt: str,
    user_prompt: str,
    response_model: type[BaseModel],
    temperature: float = 0.0,
    max_output_tokens: int = 1200,
) -> BaseModel:
    try:
        resp = client.responses.parse(
            model=model,
            input=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=temperature,
            max_output_tokens=max_output_tokens,
            text_format=response_model,
        )
        return resp.output_parsed
    except (ValidationError, Exception) as e:
        raise RuntimeError(f"Structured call failed with error: {e}")

### english translate

In [ ]:
ENGLISH_CHECK_SYSTEM = (
    "You are an expert research assistant for anthropology. "
    "Use ONLY the paragraph provided. Do NOT use outside knowledge. "
    "Return only the fields in the schema."
)

ENGLISH_CHECK_USER = (
    "Task:\n"
    "1) Determine whether the paragraph is in English.\n"
    "2) If NOT English, translate it to English. If English, keep the field empty.\n\n"
    "Rules:\n"
    "- Use ONLY the text provided.\n"
    "- non_english = 1 if not English, else 0.\n\n"
    "Paragraph:\n{paragraph}"
)


def english_check(client: OpenAI, paragraph: str, model: str = "gpt-4o-mini") -> EnglishCheckResult:
    return call_structured(
        client,
        model=model,
        system_prompt=ENGLISH_CHECK_SYSTEM,
        user_prompt=ENGLISH_CHECK_USER.format(paragraph=paragraph),
        response_model=EnglishCheckResult,
        temperature=0.0,
        max_output_tokens=1200,
    )

### first extraction

In [ ]:
DEITIES_EXTRACTION_SYSTEM = (
    "You are an expert at careful, literal information extraction from ethnographic text. "
    "Use ONLY the provided English paragraph. Do NOT use outside knowledge. "
    "Do NOT classify symbolic categories (creator/warrior/etc.) in this pass. "
    "Return only the fields in the schema."
)

DEITIES_EXTRACTION_USER = (
    "Extract all relevant elements from the paragraph:\n"
    "- Supernatural beings: deities, divine/semi-divine beings, spirits, ghosts, souls, monsters, dead ancestors (only if relevant as dead), etc.\n"
    "- Intermediaries/ritual specialists: shamans, priests, nuns, prophets, sorcerers, mediums, etc.\n"
    "- Magical/sacred objects, forces, and places.\n\n"
    "For EACH extracted element:\n"
    "- name: exact canonical name as written (or best short label from text)\n"
    "- kind: one of the schema enums\n"
    "- aliases: other names the SAME element is referred to IN THIS paragraph (only if present)\n"
    "- cat_type: individual vs multiple vs missing (ONLY if paragraph supports it)\n"
    "- evidence_deity: 1-3 short exact quotes that justify including it\n"
    "- certainty_deity: 0-100 based only on textual support\n\n"
    "Hard rules:\n"
    "- DO NOT add any categories like creator/warrior/etc. in this pass.\n"
    "- DO NOT rely on cultural background knowledge.\n\n"
    "Paragraph (English):\n{paragraph_en}"
)


def deities_extraction(paragraph_en: str, model="gpt-4o-mini") -> DeitiesExtraction:
    return call_structured(
        client,
        model=model,
        system_prompt=DEITIES_EXTRACTION_SYSTEM,
        user_prompt=DEITIES_EXTRACTION_USER.format(paragraph_en=paragraph_en),
        response_model=DeitiesExtraction,
        temperature=0.0,
        max_output_tokens=1600,
    )

In [ ]:
deities = deities_extraction(deity_text_sample)

In [ ]:
json.loads(deities.model_dump_json())

In [ ]:
deities.elements

### labelling extraction

In [ ]:
LABELLING_EXTRACTION_SYSTEM = (
    "You are an expert coder for anthropological content analysis. "
    "Use ONLY the paragraph content and the provided element list. "
    "Do NOT use outside knowledge. "
    "Only mark a category as 1 if the paragraph clearly supports it; if unsure, mark 0. "
    "Return only the fields in the schema."
)

LABELLING_EXTRACTION_USER = (
    "You will label ONLY the provided extracted elements. Do NOT add new ones.\n\n"
    "You must output:\n"
    "- inception_myth (0/1) for the paragraph\n"
    "- labels: one entry per extracted element, in the same order\n\n"
    "For each element:\n"
    "A) gender ∈ [male,female,androgynous,genderless,missing] using only paragraph clues.\n"
    "B) certainty_gender 0-100, lower if inferred indirectly.\n"
    "C) 20 binary categories (0/1) using strict textual support only:\n"
    "   cat_creator_universe: ['creator','founder','father','mother','created','conceived'] AND ['cosmos','universe','world','earth'], \n"
    "   cat_creator_human: ['creator','founder','father','mother','mould','created','made'] AND ['people','race','human','mankind','our mother','our father','made people'], \n"
    "   cat_mother: ['mother','first seed','cosmic egg','womb','her children','birthed','fertility'] OR (cat_creator_human==1 AND gender=='female'), \n"
    "   cat_wife: ['wife'], \n"
    "   cat_primal: ['primal','oldest','prime','foundation'], \n"
    "   cat_omni: ['all-wise','all-powerful','all things','all people'], \n"
    "   cat_present: ['intermediary','invocation','pay him homage','patron of','possesses','ritual','rituals'], \n"
    "   cat_absent: ['from afar','not intervening','distant','absence','contemplation','indifferent','watches from'], \n"
    "   cat_warrior: ['war','hero','military','battlefield','blade','blow','spear','blood','conquest','enemies','battle'], \n"
    "   cat_nature: ['fertility','seasons','water','animal','river'], \n"
    "   cat_cosmos: ['sky','storm','stars','sun','moon','clouds','time'], \n"
    "   cat_death: ['underworld','death','afterlife','judgement','souls','fate','ghost'], \n"
    "   cat_ruler: ['law','order','sovereign','kingship','owner'], \n"
    "   cat_dual: ['opposites','twin','embrace','neither could exist without the other'], \n"
    "   cat_trick: ['mischievous','chaos','deceiver','deceit','trick','tricks'], \n"
    "   cat_evil: ['demon','tyranny','vengeance','devil','evil','malevolent','destructive'], \n"
    "   cat_good: ['benevolent','mercy','nurturing','mediator','compassion','protection','healing','welfare','redemption','salvation'], \n"
    "   cat_demigod: ['sacred king','ancestor','supernatural','elf','elves','ghost','embodiment of the god','minor divinity','giant','apparitions','spirit','nommo','soul','monsters'], \n"
    "   cat_inter: ['shaman','shamanka','priest','priestess','nun','diviner','angel','saint','prophet','mystic','monk','oracle','magicians','lamas'], \n"
    "   cat_object_force: ['shrine','totems','church','juju','magic'], \n"
    "D) cat_type should match the element's cat_type unless you find clear contradictory evidence in the paragraph.\n"
    "E) other_names: copy aliases the element is referred to in the paragraph (from extraction), or [] if none.\n"
    "F) evidence_gender: include short exact quotes used for gender inference (or empty if missing).\n\n"
    "Hard rules:\n"
    "- If unsure on any binary category, output 0.\n"
    "- No outside knowledge.\n"
    "- Keep the same element names as provided.\n\n"
    "Extracted elements JSON:\n{elements_json}\n\n"
    "Paragraph (English):\n{translation_en}"
)


def labelling_extraction(
    client: OpenAI,
    translation_en: str,
    extracted: DeitiesExtraction,
    model: str = "gpt-4o-mini",
) -> LabellingExtraction:
    elements_json = json.dumps([e.model_dump() for e in extracted.elements], ensure_ascii=False)
    out = call_structured(
        client,
        model=model,
        system_prompt=LABELLING_EXTRACTION_SYSTEM,
        user_prompt=LABELLING_EXTRACTION_USER.format(
            elements_json=elements_json, translation_en=translation_en
        ),
        response_model=LabellingExtraction,
        temperature=0.0,
        max_output_tokens=2000,
    )

    # Optional: enforce name alignment programmatically
    extracted_names = [e.name for e in extracted.elements]
    labeled_names = [l.name for l in out.labels]
    if extracted_names != labeled_names:
        raise RuntimeError(
            "Name/order mismatch between Deities Extraction and Labelling.\n"
            f"Extracted: {extracted_names}\n"
            f"Labeled:   {labeled_names}"
        )

    return out

In [ ]:
labels = labelling_extraction(client, deity_text_sample, deities)

In [ ]:
json.loads(labels.model_dump_json())